Normally this would be an query forwarding mechanisim to a vector database

As our vector database lacks a query execution layer, the retrieval system caches all knowledge base records and performs queries internally against the cache.

Currently only cosine similarity is implemented.  Future work might make the similarity algorithm selectable, including:
- Manhattan Distance
- Euclidean Distance
- Chebyshev Distance
- etc.

In [ ]:
import logging

logger = logging.getLogger(__name__)

import heapq
from collections import deque

def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)

class RetrievalSystem:

    def __init__(self, knowledgeBase):
        self.knowledgeBase = knowledgeBase
        logger.info(f"building cache from knowledge base")
        self.cache = []
        for record in self.knowledgeBase:
            logger.debug(f"adding record for fact \"{record[0]}\" {record[1]}.")
            self.cache.append(record)
        logger.info(f"cached {len(self.cache)} records.")

    def getCosineNearest(self, vector, count):
        similarities = []
        for fact, fact_vector in self.cache:
            similarity = cosine_similarity(vector, fact_vector)
            # fill a heap to a size of count elements
            # the heap head is the smallest similarity
            if (len(similarities) < count):
                heapq.heappush(similarities, (similarity, (fact)))
            # after the heap is full, only replace on the heap if
            # the similiarity is larger than the heap head (smallest).
            else:
                if similarities[0][0] < similarity:
                    heapq.heappushpop(similarities, (similarity, (fact)))
        result = deque([])
        while len(similarities) > 0:
            # To order from largest to smallest, add to the left of the deque
            # as the popped element is guaranteed to be the smallest remaining.
            result.appendleft(heapq.heappop(similarities))
        return list(result)  